In [4]:
import os
from pathlib import Path
from typing import Optional

import fastmri
import h5py
import matplotlib.pyplot as plt
import numpy as np
import torch
from data_utils import *
from datasets import *
from fastmri.data.transforms import tensor_to_complex_np
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from torch.utils.data import DataLoader, TensorDataset

from model import *
from torch.optim import SGD, Adam, AdamW
from train_utils import *

In [5]:
# model_checkpoint_baseline = '/scratch_net/ken/mcrespo/proj_marina/logs/multivol_11_20/2024-11-21_11h25m03s/checkpoints/epoch_0499.pt'
# checkpoint_baseline = torch.load(model_checkpoint_baseline,  map_location=torch.device('cpu'))

model_checkpoint = '/scratch_net/ken/mcrespo/proj_marina/logs_new/multivol_12_23/2025-01-04_14h04m40s/checkpoints/epoch_0499.pt'
checkpoint = torch.load(model_checkpoint,  map_location=torch.device('cpu'))
model = Siren(coord_dim=3, levels=3, n_min=45, size_hashtable=12, vol_embedding_dim=256, coil_embedding_dim=128, hidden_dim=512, n_features=3, n_layers=6, out_dim=2, n_volumes=5)
model.load_state_dict(checkpoint['model_state_dict'])

/tmp/ipykernel_16499/3844117869.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(model_checkpoint,  map_location=torch.device('cpu'))


<All keys matched successfully>

In [13]:
path_to_data = '/itet-stor/mcrespo/bmicdatasets-originals/Originals/fastMRI/brain/multicoil_train/'

dataset = KCoordDataset(path_to_data, n_volumes=5, n_slices=3, with_mask=True, acceleration=4, center_frac=0.15)

coil_sizes = []
for i in range(len(dataset.metadata)):
    _, n_coils, _, _ = dataset.metadata[i]["shape"]
    coil_sizes.append(n_coils)
    
total_n_coils = torch.cumsum(torch.tensor(coil_sizes), dim=0)[-1]

# Create the indexes to access the embedding coil table
start_idx = torch.tensor([0] + list(torch.cumsum(torch.tensor(coil_sizes), dim=0)[:-1]))

# Create the table of embeddings for the coils
embeddings_coil = torch.nn.Embedding(total_n_coils.item(), 256)

embeddings_vol = torch.nn.Embedding(5, 256)

In [31]:
n_slices, n_coils, height, width = 2, 4, 320, 320
left_idx= 300
right_idx= 320
# vol_ids = torch.arange(5)
# Create tensors of indices for each dimension
kx_ids = torch.cat([torch.arange(left_idx), torch.arange(right_idx, width)])
# kx_ids = torch.arange(width)
ky_ids = torch.arange(height)
kz_ids = torch.arange(n_slices)
coil_ids = torch.arange(n_coils)

# Use meshgrid to create expanded grids
kspace_ids = torch.meshgrid(kx_ids, ky_ids, kz_ids, coil_ids, indexing="ij")
kspace_ids = torch.stack(kspace_ids, dim=-1).reshape(-1, len(kspace_ids))

dataset = TensorDataset(kspace_ids)
dataloader = DataLoader(
    dataset, batch_size=60_000, shuffle=False, num_workers=3
)

for vol_id in range(5):
    vol_embeddings = embeddings_vol(
        torch.tensor([vol_id] * 60_000, dtype=torch.long)
    )
    

    volume_kspace = torch.zeros(
        (n_slices, n_coils, height, width, 2),
        dtype=torch.float32,
    )
    print('Predicting volume')
    for point_ids in dataloader:
        point_ids = point_ids[0].long()
        coords = torch.zeros_like(
            point_ids[:,:-1], dtype=torch.float32
        )
        # Normalize the necessary coordinates for hash encoding to work
        coords[:, :2] = point_ids[:, :2]
        coords[:, 2] = (2 * point_ids[:, 2]) / (n_slices - 1) - 1
        print(coords)
        coil_embeddings = embeddings_coil(start_idx[vol_id] + point_ids[:,4])
        coords = [vol_id.repeat(len(point_ids)), coords]
    #     # Need to add `:len(coords)` because the last batch has a different size (than 60_000).
        # outputs = self.model(coords, vol_embeddings[: len(coords)], coil_embeddings)
        
    #     # "Fill in" the unsampled region.
    #     # First position corresponds to volID
    #     volume_kspace[
    #         point_ids[:, 3], point_ids[:, 4], point_ids[:, 2], point_ids[:, 1]
    #     ] = outputs

    # # Multiply by the normalization constant.
    # volume_kspace = (
    #     volume_kspace * self.dataloader.dataset.metadata[vol_id]["norm_cste"]
    # )

    # print('Volume predicted')

Predicting volume


tensor([[  0.,   0.,  -1.],
        [  0.,   0.,  -1.],
        [  0.,   0.,  -1.],
        ...,
        [ 23., 139.,   1.],
        [ 23., 139.,   1.],
        [ 23., 139.,   1.]])


IndexError: index 4 is out of bounds for dimension 1 with size 4

In [29]:
vol_id
vol_id_tensor = torch.ones(60000)*vol_id
vol_id_tensor

tensor([4., 4., 4.,  ..., 4., 4., 4.])

In [38]:
coords_list = [vol_id_tensor.unsqueeze(1), coords]
coords_v = torch.hstack(coords_list) 

In [ ]:
path_to_data = '/itet-stor/mcrespo/bmicdatasets-originals/Originals/fastMRI/brain/multicoil_train/'

dataset = KCoordDataset(path_to_data, n_volumes=n_volumes, n_slices=n_slices, with_mask=with_mask, acceleration=acceleration, center_frac=center_frac)
dataloader = DataLoader(
    dataset,
    batch_size=1_000,
    num_workers=0,
    shuffle=True,
)

vol_id = 0
shape = dataloader.dataset.metadata[vol_id]["shape"]
center_data = dataloader.dataset.metadata[vol_id]["center"]
left_idx, right_idx, center_vals = (
    center_data["left_idx"],
    center_data["right_idx"],
    center_data["vals"],
)
n_slices, n_coils, height, width = shape

volume_kspace = torch.zeros(
    (n_slices, n_coils, height, width, 2),
    dtype=torch.float32,
)
predicted_volume = volume_kspace.clone()

for batch_idx, (inputs,inputs_unnormalized,targets) in enumerate(dataloader):
    
    coords = inputs[:, 1:-1] # kx,ky,kz
    vol_ids = inputs[:,0].long()
    coil_ids = inputs[:,-1].long() # unnormalized coilID
    
    latent_vol = embedding_vol[vol_ids]
    latent_coil = embedding_coil[start_idx[vol_ids] + coil_ids]

    # outputs = model(coords, latent_vol, latent_coil)
    
    # predicted_volume[inputs_unnormalized[:,2], inputs_unnormalized[:,3], inputs_unnormalized[:,1], inputs_unnormalized[:,0]] = outputs
    volume_kspace[inputs_unnormalized[:,2], inputs_unnormalized[:,3], inputs_unnormalized[:,1], inputs_unnormalized[:,0]] = targets

In [28]:
mask = dataloader.dataset.metadata[vol_id]["mask"]
mask_squeezed = mask.squeeze(-1)
mask_squeezed = mask_squeezed.expand(shape)
unmask = 1 - mask_squeezed

zeroed = tensor_to_complex_np(volume_kspace) * mask_squeezed.numpy()
non_zeroed = tensor_to_complex_np(volume_kspace) * unmask.numpy()



In [6]:
print(psnr(gt_img, predicted_img))
print(ssim(gt_img, predicted_img))

41.181192846564926
0.9651046377258448


In [ ]:
torch.save(predicted_volume, "model_frozen_prediction_volID0.pth")

In [ ]:
gt_modulus = np.abs(fft2_shift(gt_img))
predicted_modulus = np.abs(fft2_shift(predicted_img))


plt.figure(figsize=(15,10))
plt.subplot(1,2,1)
plt.imshow(np.log(gt_modulus[0] + eps))
plt.axis('off')
plt.title('Undersampled groundtruth')
plt.colorbar()
plt.subplot(1,2,2)
plt.imshow(np.log(predicted_modulus[0] + eps))
plt.axis('off')
plt.title('Undersampled predictions')
plt.colorbar()

In [ ]:
plt.figure(figsize=(10,10))

plt.subplot(2, 2, 1)
plt.imshow(np.log(gt_modulus[0] / dataloader.dataset.metadata[vol_id]["norm_cste"] + eps))
plt.axis('off')
plt.colorbar()
plt.title("Kspace")
plt.subplot(2, 2, 2)
plt.hist(np.log(gt_modulus[0].flatten()), log=True, bins=100)
plt.subplot(2, 2, 3)
plt.imshow(np.log(predicted_modulus[0] / dataloader.dataset.metadata[vol_id]["norm_cste"] + eps))
plt.axis('off')
plt.colorbar()
plt.title("Kspace")
plt.subplot(2, 2, 4)
plt.hist(np.log(predicted_modulus[0].flatten()), log=True, bins=100)
plt.show()


In [ ]:
vol_id = 0
file = dataloader.dataset.metadata[vol_id]["file"]
with h5py.File(file, "r") as hf:
    ground_truth = hf["reconstruction_rss"][()][: n_slices]
    
modulus = np.abs(fft2_shift(ground_truth))

